### Construção das redes (TVG) e extração de features

Este notebook gera as redes de conectividade funcional a partir dos EEG pré-processados.

Fluxo:
1. Carrega `.set` em `data/preprocessed`
2. Motifs → threshold (surrogate) → TVG
3. Extrai métricas de rede
4. Salva features em `data/intermediate/sub-*_*.parquet`
5. Salva 3 CSV estilo Rosário em `data/intermediate/networks/` (`*_nodes`, `*_temporal`, `*_edges`)

Próximo passo: `02_build_dataset.ipynb` (ou `src/pipeline/build_dataset.py`) → `data/processed/dataset_features.parquet`


### Imports

In [19]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mne
import pandas as pd
from tqdm import tqdm

from src.graph.aggregate import (
    aggregate_static_network,
    edges_per_time,
    hub_occurrence,
    temporal_statistics,
    weighted_degree_static,
)
from src.graph.export_networks import save_network_exports
from src.graph.motifs import trans_motifs
from src.graph.network import graph_metrics
from src.graph.threshold import estimate_threshold
from src.graph.tvg import build_tvg_parallel


### Paths

In [20]:
SOURCE_DIR = Path("../../data/preprocessed")
OUTPUT_DIR = Path("../../data/intermediate")
NETWORKS_DIR = OUTPUT_DIR / "networks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NETWORKS_DIR.mkdir(parents=True, exist_ok=True)

print("SOURCE_DIR:", SOURCE_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("NETWORKS_DIR:", NETWORKS_DIR.resolve())


SOURCE_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\preprocessed
OUTPUT_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate
NETWORKS_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate\networks


### Parâmetros

In [21]:
WINDOW_MS = 100
MAX_LAG_MS = 25
N_SEC = 5
N_SURROGATES = 200
THRESHOLD_PERCENTILE = 99.0

### Listar arquivos EEG

In [22]:
files = sorted(SOURCE_DIR.glob("sub-*.set"))
print(f"Encontrados {len(files)} arquivos em {SOURCE_DIR}")
files[:5]

Encontrados 315 arquivos em ..\..\data\preprocessed


[WindowsPath('../../data/preprocessed/sub-010060_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010061_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010061_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EO.set')]

### Processar todos os arquivos

Para cada `.set`: motifs → threshold → TVG → métricas → parquet em `data/intermediate`.
Arquivos já existentes são pulados.


In [26]:
for file in tqdm(files, desc="Arquivos"):
    subject_id, condition = file.stem.split("_")
    stem = f"{subject_id}_{condition}"
    out_file = OUTPUT_DIR / f"{stem}.parquet"
    nodes_csv = NETWORKS_DIR / f"{stem}_nodes.csv"
    temporal_csv = NETWORKS_DIR / f"{stem}_temporal.csv"
    edges_csv = NETWORKS_DIR / f"{stem}_edges.csv"

    if out_file.exists() and nodes_csv.exists() and temporal_csv.exists() and edges_csv.exists():
        print(f"Já existe: {stem} (pulando)")
        continue

    # ==========================
    # LOAD EEG
    # ==========================
    print(f"\nProcessando {file.name}...")
    raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
    raw.pick(picks="eeg")
    ch_names = list(raw.ch_names)

    eeg = raw.get_data()
    sfreq = raw.info["sfreq"]
    eeg = eeg[:, : int(N_SEC * sfreq)]

    window_size = int(WINDOW_MS / 1000 * sfreq)
    max_lag = int(MAX_LAG_MS / 1000 * sfreq)

    # ==========================
    # MOTIFS + THRESHOLD + TVG
    # ==========================
    motifs = trans_motifs(eeg)

    threshold_info = estimate_threshold(
        motifs,
        window_size,
        max_lag,
        n_surrogates=N_SURROGATES,
        percentile=THRESHOLD_PERCENTILE,
    )
    threshold = threshold_info["threshold"]
    print(
        f"Threshold (p{THRESHOLD_PERCENTILE:.0f}): {threshold:.4f} "
        f"(nulo: μ={threshold_info['null_mean']:.4f}, "
        f"σ={threshold_info['null_std']:.4f})"
    )

    tvg = build_tvg_parallel(
        motifs=motifs,
        window_size=window_size,
        max_lag=max_lag,
        threshold=threshold,
    )

    # ==========================
    # MÉTRICAS
    # ==========================
    print("Calculando métricas de rede...")
    rea = aggregate_static_network(tvg)
    edges_t = edges_per_time(tvg)
    hub_freq = hub_occurrence(tvg)
    wdeg_static = weighted_degree_static(tvg)
    edges_stats = temporal_statistics(edges_t)
    net = graph_metrics(rea)

    features = {}
    for i, v in enumerate(net["degree"]):
        features[f"degree_{i}"] = v
    for i, v in enumerate(net["clustering"]):
        features[f"clustering_{i}"] = v
    for i, v in enumerate(net["betweenness"]):
        features[f"betweenness_{i}"] = v
    for i, v in enumerate(net["local_efficiency"]):
        features[f"local_efficiency_{i}"] = v
    for i, v in enumerate(hub_freq):
        features[f"hub_frequency_{i}"] = v
    for i, v in enumerate(wdeg_static):
        features[f"weighted_degree_{i}"] = v

    features["global_efficiency"] = net["global_efficiency"]
    features["sync_threshold"] = threshold
    features["sync_threshold_null_mean"] = threshold_info["null_mean"]
    features["sync_threshold_null_std"] = threshold_info["null_std"]
    features["edges_mean"] = edges_stats["mean"]
    features["edges_std"] = edges_stats["std"]
    features["edges_cv"] = edges_stats["cv"]

    # ==========================
    # SALVAR FEATURES + REDES
    # ==========================
    df = pd.DataFrame(features, index=[0])
    df["subject_id"] = subject_id
    df["condition"] = condition
    df.to_parquet(out_file, index=False)

    paths = save_network_exports(
        tvg=tvg,
        rea=rea,
        ch_names=ch_names,
        networks_dir=NETWORKS_DIR,
        stem=stem,
    )
    print(
        f"Salvo: {out_file.name} | TVG shape: {tvg.shape} | "
        f"redes: {paths['nodes'].name}, {paths['temporal'].name}, {paths['edges'].name}"
    )

print("Concluído.")


Arquivos:   0%|          | 1/315 [00:00<02:00,  2.61it/s]

Já existe: sub-010060_EO (pulando)
Já existe: sub-010061_EC (pulando)


Arquivos:   1%|          | 3/315 [00:00<01:28,  3.53it/s]

Já existe: sub-010061_EO (pulando)


Arquivos:   1%|▏         | 4/315 [00:01<01:29,  3.46it/s]

...

C:\Users\Anne\AppData\Local\Temp\ipykernel_22792\4196865783.py:17: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
C:\Users\Anne\AppData\Local\Temp\ipykernel_22792\4196865783.py:17: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)


Threshold (p99): 0.6324 (nulo: μ=0.4549, σ=0.0745)


Calculando métricas de rede...


Arquivos: 100%|██████████| 315/315 [09:16<00:00,  1.77s/it]

Salvo: sub-010321_EO.parquet | TVG shape: (61, 61, 1217) | redes: sub-010321_EO_nodes.csv, sub-010321_EO_temporal.csv, sub-010321_EO_edges.csv
Concluído.


### Próximo passo

Com os intermediários prontos, rode:

- Notebook: `notebook/00_redes/02_build_dataset.ipynb`
- Ou na raiz do projeto: `python src/pipeline/build_dataset.py`

Isso gera `data/processed/dataset_features.parquet` para os notebooks de EDA.

Os CSV em `data/intermediate/networks/` (`*_nodes`, `*_temporal`, `*_edges`) são exports estilo Rosário/Gephi e não entram no `build_dataset`.
